# Kashmiri Quran speech pilot
Separates likely Kashmiri speech from Arabic recitation and creates Whisper v4 draft transcripts. All outputs require human review.

In [ ]:
!nvidia-smi
!pip install -q 'transformers==4.48.3' 'peft==0.14.0' 'accelerate==1.3.0' librosa soundfile pandas


In [ ]:
from pathlib import Path
import csv, json, math, subprocess
import librosa, numpy as np, pandas as pd, soundfile as sf, torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
inputs = sorted(Path('/kaggle/input').rglob('media'))
assert inputs, 'Attached ingestion output not found'
media = inputs[0]
audio_files = sorted([p for p in media.iterdir() if p.suffix.lower() in {'.m4a','.webm','.opus','.mp3','.ogg'}])[:5]
assert audio_files, 'No source audio found'
out = Path('/kaggle/working/quran-kashmiri-speech-pilot'); clips = out/'clips'; clips.mkdir(parents=True, exist_ok=True)
print('pilot inputs:', [p.name for p in audio_files])


In [ ]:
lid_name='facebook/mms-lid-126'
lid_fx=AutoFeatureExtractor.from_pretrained(lid_name)
lid=AutoModelForAudioClassification.from_pretrained(lid_name).to('cuda').eval()
def intervals_for(y, sr=16000):
    raw=librosa.effects.split(y, top_db=32, frame_length=2048, hop_length=512)
    groups=[]; start=end=None
    for a,b in raw:
        if start is None: start,end=int(a),int(b); continue
        if a-end <= sr*0.8 and b-start <= sr*24: end=int(b)
        else:
            if end-start >= sr*2: groups.append((start,end))
            start,end=int(a),int(b)
    if start is not None and end-start >= sr*2: groups.append((start,end))
    final=[]
    for a,b in groups:
        cursor=a
        while b-cursor > sr*25: final.append((cursor,cursor+sr*25)); cursor += sr*25
        if b-cursor >= sr*2: final.append((cursor,b))
    return final
candidates=[]
for source in audio_files:
    y,_=librosa.load(source, sr=16000, mono=True)
    for index,(a,b) in enumerate(intervals_for(y)):
        candidates.append({'source':source.name,'index':index,'start':a/16000,'end':b/16000,'audio':y[a:b]})
print('speech candidates:',len(candidates))
for offset in range(0,len(candidates),8):
    batch=candidates[offset:offset+8]
    encoded=lid_fx([x['audio'] for x in batch], sampling_rate=16000, return_tensors='pt', padding=True)
    with torch.inference_mode(): probs=lid(**{k:v.to('cuda') for k,v in encoded.items()}).logits.softmax(-1).cpu()
    for row,p in zip(batch,probs):
        idx=int(p.argmax()); row['lid_label']=lid.config.id2label[idx]; row['lid_confidence']=float(p[idx])
print(pd.Series([x['lid_label'] for x in candidates]).value_counts().head(12))


In [ ]:
accepted=[x for x in candidates if x['lid_label']=='kas' and x['lid_confidence']>=0.55]
print('likely Kashmiri:',len(accepted),'hours:',round(sum(len(x['audio']) for x in accepted)/16000/3600,2))
processor=WhisperProcessor.from_pretrained('openai/whisper-large-v3-turbo',language='urdu',task='transcribe')
base=WhisperForConditionalGeneration.from_pretrained('openai/whisper-large-v3-turbo',torch_dtype=torch.float16,low_cpu_mem_usage=True).to('cuda')
model=PeftModel.from_pretrained(base,'Sameer0313/kashmiri-whisper-lora-v4-candidate').eval()
forced=processor.get_decoder_prompt_ids(language='urdu',task='transcribe')
rows=[]
for n,row in enumerate(accepted):
    features=processor(row['audio'],sampling_rate=16000,return_tensors='pt').input_features.to('cuda',dtype=torch.float16)
    with torch.inference_mode(): ids=model.generate(features,forced_decoder_ids=forced,max_new_tokens=180,num_beams=3)
    text=processor.batch_decode(ids,skip_special_tokens=True)[0].strip()
    clip_id=f'quran-ks-{n:06d}'; filename=clip_id+'.flac'
    sf.write(clips/filename,row['audio'],16000,format='FLAC')
    rows.append({'id':clip_id,'filename':filename,'draft_text':text,'source_file':row['source'],'start':round(row['start'],3),'end':round(row['end'],3),'duration':round(row['end']-row['start'],3),'lid_label':row['lid_label'],'lid_confidence':round(row['lid_confidence'],4),'review_status':'pending-native-speaker-review','training_eligible':'no'})
    if (n+1)%50==0: print('transcribed',n+1,'/',len(accepted))
fields=list(rows[0]) if rows else ['id','filename','draft_text','source_file','start','end','duration','lid_label','lid_confidence','review_status','training_eligible']
with (out/'review.csv').open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
report={'source_playlist':'PLSbPmiCa4iEEkZBX7lkLZdRdefcjLR1QH','pilot_videos':len(audio_files),'speech_candidates':len(candidates),'likely_kashmiri_clips':len(rows),'likely_kashmiri_hours':round(sum(r['duration'] for r in rows)/3600,3),'method':'energy segmentation + facebook/mms-lid-126 kas >= 0.55 + Whisper v4 draft','training_eligible':False}
(out/'report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(json.dumps(report,indent=2))
